# Setup Check: Verify RAGFlow Connection and Environment

This notebook verifies:
1. All dependencies are installed
2. Environment variables are configured
3. RAGFlow API is accessible
4. Test dataset is loaded correctly
5. OpenAI API is working

In [1]:
# Add src to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import os
from dotenv import load_dotenv

# Load environment variables
env_path = Path.cwd().parent / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print("✓ Loaded .env file")
else:
    print("⚠️  No .env file found. Copy .env.template to .env and configure it.")

✓ Loaded .env file


## 1. Check Dependencies

In [2]:
# Check all required packages
required_packages = [
    'ragas',
    'langchain',
    'langchain_openai',
    'pandas',
    'requests',
    'plotly'
]

missing = []
for pkg in required_packages:
    try:
        __import__(pkg)
        print(f"✓ {pkg}")
    except ImportError:
        print(f"✗ {pkg} - MISSING")
        missing.append(pkg)

if missing:
    print(f"\n⚠️  Missing packages: {', '.join(missing)}")
    print("Install with: pip install -r requirements.txt")
else:
    print("\n✅ All dependencies installed!")

c:\Users\junhongs\Desktop\capstone\evaluation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ ragas
✓ langchain
✓ langchain_openai
✓ pandas
✓ requests
✗ plotly - MISSING

⚠️  Missing packages: plotly
Install with: pip install -r requirements.txt


## 2. Check Environment Variables

In [ ]:
# Check required environment variables
env_vars = {
    'OPENAI_API_KEY': os.getenv('OPENAI_API_KEY'),
    'RAGFLOW_BASE_URL': os.getenv('RAGFLOW_BASE_URL', 'http://172.25.184.41:80'),
    'RAGFLOW_API_KEY': os.getenv('RAGFLOW_API_KEY'),
    'RAGFLOW_CHAT_ID': os.getenv('RAGFLOW_CHAT_ID')
}

print("Environment Variables:")
print("=" * 60)
for key, value in env_vars.items():
    if value:
        # Mask sensitive values
        if 'KEY' in key or 'API' in key:
            masked = value[:10] + '***' + value[-4:] if len(value) > 14 else '***'
            print(f"✓ {key}: {masked}")
        else:
            print(f"✓ {key}: {value}")
    else:
        print(f"✗ {key}: NOT SET")
print("=" * 60)

## 3. Test RAGFlow Connection

In [ ]:
from ragflow_client import RAGFlowClient

# Initialize client
client = RAGFlowClient()

print(f"RAGFlow Base URL: {client.base_url}")
print(f"Chat ID: {client.chat_id}")
print("\nTesting connection...")

# Test query
result = client.query(
    question="What is USO?",
    model_name="default"
)

if result['success']:
    print("\n✅ RAGFlow connection successful!")
    print(f"\nAnswer: {result['answer'][:200]}...")
    print(f"\nRetrieved contexts: {len(result['contexts'])}")
    if result['contexts']:
        print(f"First context preview: {result['contexts'][0][:150]}...")
else:
    print(f"\n✗ RAGFlow connection failed!")
    print(f"Error: {result['error']}")

## 4. Load and Validate Test Dataset

In [ ]:
from utils import load_dataset_from_jsonl, validate_dataset, print_dataset_summary

# Load dataset
dataset_path = Path.cwd().parent / 'data' / 'test_qa_pairs.jsonl'
print(f"Loading dataset from: {dataset_path}")

if not dataset_path.exists():
    print(f"\n✗ Dataset file not found!")
    print(f"Expected location: {dataset_path}")
else:
    dataset = load_dataset_from_jsonl(str(dataset_path))
    
    # Validate
    is_valid = validate_dataset(dataset)
    
    if is_valid:
        print(f"\n✅ Dataset is valid!")
        print_dataset_summary(dataset)
        
        # Show first example
        print("\nFirst Test Case:")
        print("=" * 60)
        print(f"Question: {dataset[0]['user_input']}")
        print(f"\nGround Truth: {dataset[0]['reference'][:200]}...")
        print(f"\nReference Contexts: {len(dataset[0]['reference_contexts'])} contexts")
        print("=" * 60)
    else:
        print(f"\n✗ Dataset validation failed!")

## 5. Test OpenAI API (for Ragas Evaluation)

In [ ]:
from langchain_openai import ChatOpenAI

try:
    # Test OpenAI API
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    response = llm.invoke("Say 'OpenAI API is working!'")
    print("✅ OpenAI API connection successful!")
    print(f"Response: {response.content}")
except Exception as e:
    print(f"✗ OpenAI API connection failed!")
    print(f"Error: {e}")

## 6. Review Embedding Models Configuration

In [ ]:
from config import EMBEDDING_MODELS
import pandas as pd

# Display models
models_df = pd.DataFrame(EMBEDDING_MODELS)
print("\nEmbedding Models to Evaluate:")
print("=" * 80)
print(models_df[['display_name', 'provider', 'model', 'dimensions']].to_string(index=False))
print("=" * 80)

## Summary

If all checks pass, you're ready to run the evaluation!

Next steps:
1. Run `01_single_model_test.ipynb` to test with one model
2. Run `02_multi_model_eval.ipynb` for full multi-model comparison